# Latent Ablation Study: Turning Off Latent Layers

In this notebook, we take a trained NVAE model and turn off some latent layers during reconstruction and generation, to see how it deteriorates the reconstruction and generation quality.

This is a more rigorous analysis of posterior collapse. First, we can take a look at the KL divergence train graphs for each layer. If the KL divergence is close to 0, it is an indicator of potential posterior collapse. To verify this, we can turn off this layer and see if it affects evaluation performance.

## Preliminaries

The repository assumes all entry points to be run as modules from root. Since
this notebook is 2 levels deep, the below block moves us to the root directory.

In [ ]:
import os

# Change this variable if the root folder name has been changed
root_dir = "nvae-shape-encoding"
current_dir = os.getcwd()

if not current_dir.endswith(root_dir):
    %cd ../..

assert os.getcwd().endswith(root_dir)

Update `model_path` to choose the model.

In [ ]:
import lightning as L
import torch

from arch.nvae.nvae import NVAE
from utils.const import SEED
from data_modules.acdc import ACDCMaskDataModule
from utils.utils import setup_device

model_path = "logs/nvae_acdc/best/default/checkpoints/epoch=97-step=20972.ckpt"

# Setup device
device = setup_device()
print(f"Device: {device}")

# Seed
L.seed_everything(SEED)

# Load data
data_module = ACDCMaskDataModule(batch_size=20)

# Reseed after preprocessing data
L.seed_everything(SEED)

# Load model
model = NVAE.load_from_checkpoint(model_path)

## Reconstruction

First, let's look at reconstruction quality.

NVAE has a shared encoder-decoder architecture. During the top-down pass, the encoder shares information with the decoder to build the residual distribution at each layer. Here, by turning off a layer, we prevent information from being shared.

In [ ]:
from utils.utils import get_data

loader_test = data_module.test_dataloader(shuffle=True)
x = get_data(loader_test)
x.shape

In [ ]:

from utils.utils import discretise, show_samples

num_data = x.shape[0]
samples_idx = torch.randperm(num_data)[:6]
x_batch = x[samples_idx]
samples = [torch.argmax(x_batch, dim=1).unsqueeze(1)]

with torch.no_grad():
    model.eval()
    model.to(device)
    x_batch = x_batch.to(device)
    
    for i in range(3, 0, -1):
        print("Number of shared layers:", i)
        x_hat_logits, _, _, _, _ = model(x_batch, test=True, num_shared_layers=i)
        
        # Collect sample to visualise
        x_hat_logits = x_hat_logits.cpu()
        reconstructions = torch.argmax(x_hat_logits, dim=1).unsqueeze(1)
        samples.append(reconstructions)

len(samples)

In [ ]:
import matplotlib.pyplot as plt
from torchvision.utils import make_grid

from utils.colourmap import GIRIDIS

fig, ax = plt.subplots(1, 4, figsize=(24, 36))

titles = [
    "Ground Truth",
    "3 Active Layers",
    "2 Active Layers",
    "1 Active Layer",
]

for i, reconstructions in enumerate(samples):
    reconstructions = reconstructions.cpu().float()
    samples_grid = make_grid(reconstructions, nrow=1, normalize=True)
    samples_grid = samples_grid[0]
    
    ax[i].axis("off")
    ax[i].imshow(samples_grid, cmap=GIRIDIS)
    ax[i].set_title(titles[i], fontsize=36, pad=36)

## Generation

Now, let's look at generation quality.

Here, by turning off a layer, we do not sample from its corresponding Normal distribution, but instead take the mean as the latent variable.

Example: Say we set `num_sample_layers = 1`. This means the only variation comes from the topmost latent layer. If we observe the generated samples to be very similar, it means the topmost layer does not encode much information and has collapsed.

In [ ]:
from utils.anatomical_validity_checker import AnatomicalValidityChecker
from utils.const import FRDS_MODEL_PATH
from utils.eval import compute_frds


num_test_data, _, _, _ = x.shape

samples = []

with torch.no_grad():
    model.eval()
    model.to(device)

    # Generate probabilistic segmentation maps
    z_shape = model.decoder.get_top_latent_shape(batch_size=num_test_data)
    z = torch.randn(z_shape, device=device)
    
    for i in range(3, 0, -1):
        x_fake = model.decoder.generate(
            num_samples=num_test_data,
            device=device,
            num_sample_layers=i,
            z=z,
        )
                
        feats_fake = model.conditional_coder(x_fake)
        
        discretised_feats_fake = discretise(feats_fake)
        
        frds_value = compute_frds(
            x,
            discretised_feats_fake,
            resnet_path=FRDS_MODEL_PATH,
            device=device,
        )

        print(f"FRDS for {i} layers: {frds_value}")
        
        # Percentage of anatomically valid generations
        num_valid = 0
        
        for discretised_feat_fake in discretised_feats_fake:
            AV = AnatomicalValidityChecker(discretised_feat_fake)
            if AV.count_violations() == 0:
                num_valid += 1
        
        print(f"Anatomically valid for {i} layers: {num_valid / num_test_data}")

        # Discretise probabilistic map then view generations
        generations = torch.argmax(feats_fake[:6], dim=1).unsqueeze(1)
        samples.append(generations)

len(samples)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 36))

titles = [
    "3 Active Layers",
    "2 Active Layers",
    "1 Active Layer",
]

for i, generations in enumerate(samples):
    generations = generations.cpu().float()
    samples_grid = make_grid(generations, nrow=1, normalize=True)
    samples_grid = samples_grid[0]
    
    ax[i].axis("off")
    ax[i].imshow(samples_grid, cmap=GIRIDIS)
    ax[i].set_title(titles[i], fontsize=36, pad=36)